In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila


In [3]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Drosophila_chemical_protein
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Drosophila_chemical_protein/Drosophila_chemical_protein.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

In [4]:
#

# bindingdb

In [5]:

bindingdb = pd.read_csv(PROC_DIR + 'bindingdb/Chemical_Protein_Droso_BindingDB.csv')

bindingdb.columns = bindingdb.columns.str.lower()
bindingdb['kg_type'] = 'Generalised'
bindingdb['kg_source'] = 'BindingDB'
bindingdb['species'] = 'D.melanogaster'
print(f"bindingdb: {len(bindingdb):,} rows")
bindingdb

bindingdb: 12 rows


,head,relation,tail,head_type,tail_type,kg_source,species,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,124189,ChemicalEntity_Protein,P53624,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(1R,2R,3R,4S,5R)-4-amino-5-methylsulfanylcyclo...",CS[C@@H]1[C@H]([C@H]([C@H]([C@H]1O)O)O)N,"Mannosyl-oligosaccharide alpha-1,2-mannosidase...",Pubchem,Uniprot_ID,Generalised
1,11987837,ChemicalEntity_Protein,Q24451,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(1R,2R,3S,4S)-5-aminocyclopentane-1,2,3,4-tetrol",[C@H]1([C@@H]([C@@H](C([C@@H]1O)N)O)O)O,Alpha-mannosidase 2,Pubchem,Uniprot_ID,Generalised
2,128174,ChemicalEntity_Protein,Q24451,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(1R,2R,3R,4S,5R)-4-amino-5-methylsulfinylcyclo...",CS(=O)[C@@H]1[C@H]([C@H]([C@H]([C@H]1O)O)O)N,Alpha-mannosidase 2,Pubchem,Uniprot_ID,Generalised
3,11593442,ChemicalEntity_Protein,Q24451,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(1S,2S,3R,4R)-4-aminocyclopentane-1,2,3-triol",C1[C@H]([C@H]([C@H]([C@H]1O)O)O)N,Alpha-mannosidase 2,Pubchem,Uniprot_ID,Generalised
4,11636858,ChemicalEntity_Protein,Q24451,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(1R,2R,3R,4S,5R)-4-amino-5-methoxycyclopentane...",CO[C@@H]1[C@H]([C@H]([C@H]([C@H]1O)O)O)N,Alpha-mannosidase 2,Pubchem,Uniprot_ID,Generalised
5,44322824,ChemicalEntity_Protein,P15348,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(1S,16R)-7,16-dihydroxy-5,14-dimethyl-2-oxa-14...",CC1=CC(=C2C(=C1)O[C@]34[C@@H](C(CC=C3C2=O)N(C4...,DNA topoisomerase 2 {ECO:0000305},Pubchem,Uniprot_ID,Generalised
6,44322826,ChemicalEntity_Protein,P15348,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(4R,4aS)-4,8-dihydroxy-N,6-dimethyl-9-oxo-4H-x...",CC1=CC(=C2C(=C1)O[C@@]3([C@@H](C=CC=C3C2=O)O)C...,DNA topoisomerase 2 {ECO:0000305},Pubchem,Uniprot_ID,Generalised
7,51351571,ChemicalEntity_Protein,P53624,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(2R,3S,4S,5S,6R)-2-(hydroxymethyl)-6-(2-phenyl...",C1=CC=C(C=C1)CCS(=O)[C@@H]2[C@H]([C@H]([C@@H](...,"Mannosyl-oligosaccharide alpha-1,2-mannosidase...",Pubchem,Uniprot_ID,Generalised
8,51351570,ChemicalEntity_Protein,P53624,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(2R,3S,4S,5S,6R)-2-benzylsulfonyl-6-(hydroxyme...",C1=CC=C(C=C1)CS(=O)(=O)[C@@H]2[C@H]([C@H]([C@@...,"Mannosyl-oligosaccharide alpha-1,2-mannosidase...",Pubchem,Uniprot_ID,Generalised
9,51351561,ChemicalEntity_Protein,P53624,ChemicalEntity,Protein,BindingDB,D.melanogaster,"(2R,3S,4S,5S,6R)-2-(hydroxymethyl)-6-(2-phenyl...",C1=CC=C(C=C1)CCS(=O)(=O)[C@@H]2[C@H]([C@H]([C@...,"Mannosyl-oligosaccharide alpha-1,2-mannosidase...",Pubchem,Uniprot_ID,Generalised


# Consolidate into Unified Schem

In [6]:
# List all source DataFrames to include
source_dfs = [
    bindingdb
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 12


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,124189,ChemicalEntity_Protein,P53624,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1R,2R,3R,4S,5R)-4-amino-5-methylsulfanylcyclo...","Mannosyl-oligosaccharide alpha-1,2-mannosidase...",D.melanogaster
1,11987837,ChemicalEntity_Protein,Q24451,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1R,2R,3S,4S)-5-aminocyclopentane-1,2,3,4-tetrol",Alpha-mannosidase 2,D.melanogaster
2,128174,ChemicalEntity_Protein,Q24451,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1R,2R,3R,4S,5R)-4-amino-5-methylsulfinylcyclo...",Alpha-mannosidase 2,D.melanogaster
3,11593442,ChemicalEntity_Protein,Q24451,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1S,2S,3R,4R)-4-aminocyclopentane-1,2,3-triol",Alpha-mannosidase 2,D.melanogaster
4,11636858,ChemicalEntity_Protein,Q24451,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1R,2R,3R,4S,5R)-4-amino-5-methoxycyclopentane...",Alpha-mannosidase 2,D.melanogaster
5,44322824,ChemicalEntity_Protein,P15348,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1S,16R)-7,16-dihydroxy-5,14-dimethyl-2-oxa-14...",DNA topoisomerase 2 {ECO:0000305},D.melanogaster
6,44322826,ChemicalEntity_Protein,P15348,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(4R,4aS)-4,8-dihydroxy-N,6-dimethyl-9-oxo-4H-x...",DNA topoisomerase 2 {ECO:0000305},D.melanogaster
7,51351571,ChemicalEntity_Protein,P53624,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(2R,3S,4S,5S,6R)-2-(hydroxymethyl)-6-(2-phenyl...","Mannosyl-oligosaccharide alpha-1,2-mannosidase...",D.melanogaster
8,51351570,ChemicalEntity_Protein,P53624,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(2R,3S,4S,5S,6R)-2-benzylsulfonyl-6-(hydroxyme...","Mannosyl-oligosaccharide alpha-1,2-mannosidase...",D.melanogaster
9,51351561,ChemicalEntity_Protein,P53624,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(2R,3S,4S,5S,6R)-2-(hydroxymethyl)-6-(2-phenyl...","Mannosyl-oligosaccharide alpha-1,2-mannosidase...",D.melanogaster


# Sanity Check — Distinct Values

In [7]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'ChemicalEntity_Protein'}
head_type           : {'ChemicalEntity'}
tail_type           : {'Protein'}
relation_type       : {None}
kg_source           : {'BindingDB'}
head_id_is          : {'Pubchem'}
tail_id_is          : {'Uniprot_ID'}


In [8]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 12 remaining


# NaN Audit (pre-dedup)

In [9]:
true_nan   = final_df.isna().sum()
genDR_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_genDR_count": genDR_nan,
    'Total_NaN_like':     true_nan + genDR_nan
})

,NaN_count,'NAN'_genDR_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,12,0,12
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [10]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 12  |  After dedup: 12


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,124189,ChemicalEntity_Protein,P53624,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1R,2R,3R,4S,5R)-4-amino-5-methylsulfanylcyclo...","Mannosyl-oligosaccharide alpha-1,2-mannosidase..."
1,128174,ChemicalEntity_Protein,Q24451,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1R,2R,3R,4S,5R)-4-amino-5-methylsulfinylcyclo...",Alpha-mannosidase 2
2,11593442,ChemicalEntity_Protein,Q24451,ChemicalEntity,None,Protein,BindingDB,Generalised,Pubchem,Uniprot_ID,"(1S,2S,3R,4R)-4-aminocyclopentane-1,2,3-triol",Alpha-mannosidase 2


In [11]:
true_nan   = final_df_dedup.isna().sum()
genDR_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_genDR_count": genDR_nan,
    'Total_NaN_like':     true_nan + genDR_nan
})

,NaN_count,'NAN'_genDR_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,12,0,12
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [12]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'BindingDB'} kg_source
BindingDB    12
Name: count, dtype: int64


In [13]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    12
Name: count, dtype: int64


In [14]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 12 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Drosophila_chemical_protein/Drosophila_chemical_protein.csv


In [15]:
#